# Task 6: Consensus Clustering, Algorithm Comparison and Baselines

This notebook conducts consensus clustering, algorithmic comparisons, alternative embedding baselines, and chunk-averaging sensitivity checks on the constitutional preambles dataset:
1. **Consensus Clustering (Six Versions)**: 200 random KMeans runs ($k=7$) per embedding representation, co-association matrix generation, agglomerative hierarchical consensus extraction (average linkage with fallback to complete linkage if cluster size $<5$), split-half reproducibility verification (seeds 0..99 vs 100..199), and $6 	imes 6$ consensus agreement evaluation.
2. **Representation C_c Deep-Dive**: Renumbering consensus clusters from largest to smallest ($0..6$), per-country cluster confidence and runner-up metrics, cluster membership catalogs, lowest-confidence audit, contingency analysis against original clusters, and length profiles.
3. **Algorithm Comparison on C_c**: Evaluating Agglomerative clustering (Ward and Average Cosine), Gaussian Mixture Models on 10 principal components (Spherical and Diagonal covariance with BIC evaluation across $k \in [2, 12]$), and HDBSCAN density clustering ($min\_cluster\_size \in \{5, 8\}$).
4. **Embedding Baselines on Masked Text**: Comparing $C_c$ (MPNet) against a lexical baseline $T_c$ (TF-IDF + SVD-50) and a lightweight transformer baseline $M_c$ (all-MiniLM-L6-v2 with 200-token chunking).
5. **Chunk-Averaging Sensitivity ($D_c$)**: Evaluating truncation to the first 300 tokens without chunk averaging on MPNet to assess whether multi-chunk averaging alters cluster assignment.
6. **Consensus Dataset Export**: Persisting all consensus labelings, confidence metrics, and baseline assignments to `data/processed/cluster_consensus.csv`.

## 1. Environment, Input Verification, and Embedding Setup
Load `data/processed/final_country_order.csv`, `emb_final_raw.npy` (A), `emb_final_clean.npy` (B), and `emb_final_masked.npy` (C). Construct column-centered and row L2-normalized representations ($A_c, B_c, C_c$), verify shapes and country alignments, and load existing single-run labels from `cluster_runs.csv`.

In [1]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import kruskal
from sklearn.cluster import KMeans, AgglomerativeClustering, HDBSCAN
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sentence_transformers import SentenceTransformer

data_dir = '../data/processed'

# 1. Load order and raw data
df_order = pd.read_csv(os.path.join(data_dir, 'final_country_order.csv'))
df_runs = pd.read_csv(os.path.join(data_dir, 'cluster_runs.csv'))
df_final = pd.read_csv(os.path.join(data_dir, 'preamble_final.csv'))
df_clusterable = df_final[df_final['status'].isin(['preamble_ok', 'preamble_cut'])].copy().reset_index(drop=True)

assert len(df_order) == 158, f"Expected 158 rows, got {len(df_order)}"
assert (df_order['Country'] == df_clusterable['Country']).all(), "Country alignment mismatch with preamble_final.csv"
assert (df_order['Country'] == df_runs['Country']).all(), "Country alignment mismatch with cluster_runs.csv"
print("Input verification passed: 158 countries strictly aligned across order, preamble, and run files.")

# 2. Load embeddings
emb_A = np.load(os.path.join(data_dir, 'emb_final_raw.npy'))
emb_B = np.load(os.path.join(data_dir, 'emb_final_clean.npy'))
emb_C = np.load(os.path.join(data_dir, 'emb_final_masked.npy'))

def center_and_normalize(emb):
    centered = emb - emb.mean(axis=0)
    norm = np.linalg.norm(centered, axis=1, keepdims=True)
    return centered / norm

emb_A_c = center_and_normalize(emb_A)
emb_B_c = center_and_normalize(emb_B)
emb_C_c = center_and_normalize(emb_C)

assert emb_A.shape == (158, 768) and emb_B.shape == (158, 768) and emb_C.shape == (158, 768)
assert emb_A_c.shape == (158, 768) and emb_B_c.shape == (158, 768) and emb_C_c.shape == (158, 768)

versions = {
    'A': emb_A, 'B': emb_B, 'C': emb_C,
    'A_c': emb_A_c, 'B_c': emb_B_c, 'C_c': emb_C_c
}

print(f"All six embedding representations initialized successfully with shape (158, 768).")


/home/devpanchal/Projects/PreambleMap/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input verification passed: 158 countries strictly aligned across order, preamble, and run files.
All six embedding representations initialized successfully with shape (158, 768).


## 2. Consensus Clustering Across All Six Versions
For each of the six embedding representations ($A, B, C, A_c, B_c, C_c$):
1. Execute 200 iterations of KMeans ($k=7, n_{	ext{init}}=10$) across seeds $0..199$ and compute the $158 	imes 158$ co-association matrix $P$.
2. Form distance matrix $D = 1 - P$ and apply Agglomerative Clustering ($k=7$, precomputed metric, average linkage). If the smallest cluster has $< 5$ countries, evaluate complete linkage and select the linkage yielding the larger minimum cluster size.
3. Assess split-half reproducibility by comparing consensus partitions derived independently from seeds $0..99$ vs. $100..199$ via Adjusted Rand Index (ARI).
4. Report cluster sizes, ARI against single-run solutions ($n_{	ext{init}}=50, 	ext{random\_state}=42$), ARI against original clusters, and cosine silhouette.
5. Display the $6 	imes 6$ ARI matrix across consensus labelings.

In [2]:
def get_consensus_labels(emb, n_seeds=200):
    P = np.zeros((len(emb), len(emb)), dtype=np.float64)
    for s in range(n_seeds):
        km = KMeans(n_clusters=7, n_init=10, random_state=s)
        labs = km.fit_predict(emb)
        P += (labs[:, None] == labs[None, :])
    P /= n_seeds
    D = 1.0 - P
    
    agg_avg = AgglomerativeClustering(n_clusters=7, metric='precomputed', linkage='average')
    labs_avg = agg_avg.fit_predict(D)
    min_size_avg = pd.Series(labs_avg).value_counts().min()
    
    if min_size_avg < 5:
        agg_comp = AgglomerativeClustering(n_clusters=7, metric='precomputed', linkage='complete')
        labs_comp = agg_comp.fit_predict(D)
        min_size_comp = pd.Series(labs_comp).value_counts().min()
        if min_size_comp > min_size_avg:
            return labs_comp, P, 'complete', (min_size_avg, min_size_comp)
        else:
            return labs_avg, P, 'average', (min_size_avg, min_size_comp)
    else:
        return labs_avg, P, 'average', (min_size_avg, None)

print("Computing consensus clustering (200 seeds) across all 6 representations...")
cons_labels_dict = {}
P_dict = {}
linkage_used = {}

for v_name, emb in versions.items():
    labs, P, link, sizes = get_consensus_labels(emb, 200)
    cons_labels_dict[v_name] = labs
    P_dict[v_name] = P
    linkage_used[v_name] = (link, sizes)

# Reproducibility split (0..99 vs 100..199)
split_ari = {}
for v_name, emb in versions.items():
    labs_1, _, _, _ = get_consensus_labels(emb, 100)
    
    P_2 = np.zeros((len(emb), len(emb)), dtype=np.float64)
    for s in range(100, 200):
        km = KMeans(n_clusters=7, n_init=10, random_state=s)
        labs_s = km.fit_predict(emb)
        P_2 += (labs_s[:, None] == labs_s[None, :])
    P_2 /= 100.0
    D_2 = 1.0 - P_2
    
    agg = AgglomerativeClustering(n_clusters=7, metric='precomputed', linkage=linkage_used[v_name][0])
    labs_2 = agg.fit_predict(D_2)
    split_ari[v_name] = adjusted_rand_score(labs_1, labs_2)

# Summary table
summary_records = []
for v_name in versions:
    labs = cons_labels_dict[v_name]
    emb = versions[v_name]
    single_labs = df_runs[f'labels_{v_name}']
    orig_clusters = df_order['orig_cluster']
    
    ari_single = adjusted_rand_score(labs, single_labs)
    ari_orig = adjusted_rand_score(labs, orig_clusters)
    sil = silhouette_score(emb, labs, metric='cosine')
    
    summary_records.append({
        'version': v_name,
        'linkage': linkage_used[v_name][0],
        'cluster_sizes': list(pd.Series(labs).value_counts().sort_values(ascending=False).values),
        'ari_single': ari_single,
        'ari_orig': ari_orig,
        'silhouette': sil,
        'split_ari': split_ari[v_name]
    })

df_cons_summary = pd.DataFrame(summary_records)
print("\n=== CONSENSUS CLUSTERING SUMMARY (ALL SIX VERSIONS) ===")
display(df_cons_summary)

# 6x6 consensus ARI matrix
ver_keys = ['A', 'B', 'C', 'A_c', 'B_c', 'C_c']
ari_6x6_cons = np.zeros((6, 6))
for i, v1 in enumerate(ver_keys):
    for j, v2 in enumerate(ver_keys):
        ari_6x6_cons[i, j] = adjusted_rand_score(cons_labels_dict[v1], cons_labels_dict[v2])
df_ari_6x6_cons = pd.DataFrame(ari_6x6_cons, index=ver_keys, columns=ver_keys)

print("\n=== 6x6 ARI MATRIX BETWEEN CONSENSUS LABELINGS ===")
display(df_ari_6x6_cons.round(4))


Computing consensus clustering (200 seeds) across all 6 representations...



=== CONSENSUS CLUSTERING SUMMARY (ALL SIX VERSIONS) ===


,version,linkage,cluster_sizes,ari_single,ari_orig,silhouette,split_ari
0,A,average,"[29, 29, 24, 24, 20, 18, 14]",0.672667,0.569961,0.121142,0.811079
1,B,complete,"[38, 29, 26, 19, 16, 16, 14]",0.678151,0.722584,0.118290,0.978249
2,C,average,"[55, 29, 18, 17, 17, 15, 7]",0.492614,0.345765,0.050209,0.760281
3,A_c,average,"[28, 28, 24, 22, 21, 20, 15]",0.799098,0.570938,0.144414,0.968015
4,B_c,average,"[36, 29, 24, 23, 20, 16, 10]",0.834649,0.743283,0.152405,0.883469
5,C_c,average,"[30, 27, 25, 23, 22, 16, 15]",0.950466,0.472743,0.159549,0.955222



=== 6x6 ARI MATRIX BETWEEN CONSENSUS LABELINGS ===


,A,B,C,A_c,B_c,C_c
A,1.0000,0.6010,0.4898,0.8389,0.6065,0.5636
B,0.6010,1.0000,0.3683,0.6396,0.8839,0.5531
C,0.4898,0.3683,1.0000,0.3995,0.3544,0.6156
A_c,0.8389,0.6396,0.3995,1.0000,0.6797,0.6064
B_c,0.6065,0.8839,0.3544,0.6797,1.0000,0.5610
C_c,0.5636,0.5531,0.6156,0.6064,0.5610,1.0000


## 3. Detailed Analysis of Variant C_c Consensus
For Variant $C_c$, renumber clusters from largest to smallest ($0 =$ largest, $6 =$ smallest) and maintain this convention throughout. Evaluate:
- Per-country cluster confidence (mean co-association $P_{ij}$ with all other members of its own cluster).
- Runner-up cluster (alternative cluster with highest average co-association) and runner-up score.
- Mean cluster confidence.
- Complete cluster roster sorted by confidence.
- 25 lowest-confidence countries.
- Contingency crosstab against original clusters.
- Word count distributions ($n$, median, min, max) per cluster.

In [3]:
# Renumber C_c consensus labels 0=largest to 6=smallest
raw_Cc_labs = cons_labels_dict['C_c']
P_Cc = P_dict['C_c']
counts_Cc = pd.Series(raw_Cc_labs).value_counts()
mapping_Cc = {old: new for new, old in enumerate(counts_Cc.index)}
cons_C_c = np.array([mapping_Cc[c] for c in raw_Cc_labs])
cons_labels_dict['C_c'] = cons_C_c

# Confidence and runner-up
conf_Cc = []
runnerup_Cc = []
runnerup_score_Cc = []

for i in range(len(df_order)):
    c_i = cons_C_c[i]
    same_m = [j for j in range(len(df_order)) if cons_C_c[j] == c_i and j != i]
    conf_Cc.append(P_Cc[i, same_m].mean())
    
    best_k = None
    best_score = -1.0
    for k in range(7):
        if k == c_i: continue
        k_m = [j for j in range(len(df_order)) if cons_C_c[j] == k]
        k_score = P_Cc[i, k_m].mean()
        if k_score > best_score:
            best_score = k_score
            best_k = k
    runnerup_Cc.append(best_k)
    runnerup_score_Cc.append(best_score)

conf_Cc = np.array(conf_Cc)
runnerup_Cc = np.array(runnerup_Cc)
runnerup_score_Cc = np.array(runnerup_score_Cc)

print("=== C_c CONSENSUS CLUSTER SIZES (0=LARGEST) ===")
print(pd.Series(cons_C_c).value_counts().sort_index())

print("\n=== MEAN CONFIDENCE PER CLUSTER (C_c) ===")
for k in range(7):
    print(f"  Cluster {k} (n={(cons_C_c == k).sum()}): {conf_Cc[cons_C_c == k].mean():.4f}")

df_country_conf = pd.DataFrame({
    'Country': df_order['Country'],
    'cluster': cons_C_c,
    'confidence': conf_Cc,
    'runner_up': runnerup_Cc,
    'runner_up_score': runnerup_score_Cc
})

print("\n=== ALL MEMBER COUNTRIES PER CLUSTER SORTED BY CONFIDENCE ===")
for k in range(7):
    print(f"\n--- Cluster {k} (n={(cons_C_c == k).sum()}, mean conf = {conf_Cc[cons_C_c == k].mean():.4f}) ---")
    sub = df_country_conf[df_country_conf['cluster'] == k].sort_values(by='confidence', ascending=False)
    for _, r in sub.iterrows():
        print(f"  {r['Country']:<25} Conf: {r['confidence']:.4f}  |  Runner-up: Cluster {r['runner_up']} (score: {r['runner_up_score']:.4f})")

print("\n=== 25 LOWEST-CONFIDENCE COUNTRIES (C_c) ===")
lowest_25 = df_country_conf.sort_values(by='confidence', ascending=True).head(25)
display(lowest_25)

print("\n=== CROSSTAB: C_c CONSENSUS VS. ORIGINAL CLUSTERS ===")
ct_Cc = pd.crosstab(cons_C_c, df_order['orig_cluster'], rownames=['Consensus C_c'], colnames=['Original Cluster'])
display(ct_Cc)

print("\n=== WORD COUNT DISTRIBUTIONS PER CLUSTER (C_c) ===")
words = df_order['extracted_words'].values
c_len_stats = []
for k in range(7):
    kw = words[cons_C_c == k]
    c_len_stats.append({
        'cluster': k,
        'n': len(kw),
        'median': np.median(kw),
        'min': np.min(kw),
        'max': np.max(kw)
    })
df_c_stats = pd.DataFrame(c_len_stats)
display(df_c_stats)


=== C_c CONSENSUS CLUSTER SIZES (0=LARGEST) ===
0    30
1    27
2    25
3    23
4    22
5    16
6    15
Name: count, dtype: int64

=== MEAN CONFIDENCE PER CLUSTER (C_c) ===
  Cluster 0 (n=30): 0.7555
  Cluster 1 (n=27): 0.7875
  Cluster 2 (n=25): 0.8899
  Cluster 3 (n=23): 0.8417
  Cluster 4 (n=22): 0.6654
  Cluster 5 (n=16): 0.8990
  Cluster 6 (n=15): 0.7445

=== ALL MEMBER COUNTRIES PER CLUSTER SORTED BY CONFIDENCE ===

--- Cluster 0 (n=30, mean conf = 0.7555) ---
  Ghana                     Conf: 0.8500  |  Runner-up: Cluster 4 (score: 0.0573)
  Mongolia                  Conf: 0.8500  |  Runner-up: Cluster 6 (score: 0.0580)
  Nauru                     Conf: 0.8500  |  Runner-up: Cluster 4 (score: 0.0573)
  United States of America  Conf: 0.8500  |  Runner-up: Cluster 6 (score: 0.0580)
  Tajikistan                Conf: 0.8500  |  Runner-up: Cluster 6 (score: 0.0580)
  Kiribati                  Conf: 0.8500  |  Runner-up: Cluster 6 (score: 0.0580)
  Zambia                    Conf: 0.8

,Country,cluster,confidence,runner_up,runner_up_score
51,Ethiopia,4,0.257857,0,0.306167
2,Algeria,1,0.317692,5,0.659687
85,Madagascar,3,0.371818,4,0.235682
73,Japan,6,0.382857,0,0.342000
88,Marshall Islands,0,0.395000,4,0.466136
115,Saint Lucia,4,0.436190,1,0.399074
5,Antigua and Barbuda,4,0.439762,1,0.394630
126,S. Sudan,6,0.490357,0,0.277000
37,Costa Rica,0,0.509310,3,0.288261
106,Papua New Guinea,4,0.518095,1,0.223704



=== CROSSTAB: C_c CONSENSUS VS. ORIGINAL CLUSTERS ===


Original Cluster,0,1,2,3,4,5,6
Consensus C_c,,,,,,,
0,6,5,0,4,5,0,10
1,0,0,1,0,1,0,25
2,0,10,0,1,0,14,0
3,17,1,0,1,4,0,0
4,2,0,0,0,19,0,1
5,0,0,15,1,0,0,0
6,0,0,1,13,1,0,0



=== WORD COUNT DISTRIBUTIONS PER CLUSTER (C_c) ===


,cluster,n,median,min,max
0,0,30,114.0,33,376
1,1,27,308.0,95,1307
2,2,25,118.0,59,685
3,3,23,182.0,33,610
4,4,22,240.0,70,2011
5,5,16,479.0,165,3223
6,6,15,385.0,186,1160


## 4. Alternative Clustering Algorithm Comparison on $C_c$
Evaluate alternative clustering algorithms on representation $C_c$ at $k=7$ (unless dictated by the algorithm):
1. Agglomerative Clustering (Ward linkage).
2. Agglomerative Clustering (Average linkage, cosine metric).
3. Gaussian Mixture Models on the first 10 principal components of $C_c$ (Spherical and Diagonal covariance, $n_{	ext{init}}=20, 	ext{random\_state}=42$, with BIC evaluated across $k \in [2, 12]$).
4. HDBSCAN density clustering ($min\_cluster\_size \in \{5, 8\}$).

Metrics reported: cluster sizes, cosine silhouette (excluding noise points for HDBSCAN), ARI against consensus $C_c$, and ARI against single-run $C_c$ ($n_{	ext{init}}=50, 	ext{random\_state}=42$).

In [4]:
algo_results = []

# a. Ward
ward = AgglomerativeClustering(n_clusters=7, linkage='ward')
labs_ward = ward.fit_predict(emb_C_c)
algo_results.append({
    'algorithm': 'Agglomerative (Ward)',
    'k': 7,
    'cluster_sizes': list(pd.Series(labs_ward).value_counts().sort_values(ascending=False).values),
    'silhouette': silhouette_score(emb_C_c, labs_ward, metric='cosine'),
    'ari_cons_Cc': adjusted_rand_score(labs_ward, cons_C_c),
    'ari_single_Cc': adjusted_rand_score(labs_ward, df_runs['labels_C_c']),
    'noise_points': 0,
    'ari_cons_Cc_non_noise': adjusted_rand_score(labs_ward, cons_C_c)
})

# b. Average cosine
avg_cos = AgglomerativeClustering(n_clusters=7, metric='cosine', linkage='average')
labs_avg_cos = avg_cos.fit_predict(emb_C_c)
algo_results.append({
    'algorithm': 'Agglomerative (Average, Cosine)',
    'k': 7,
    'cluster_sizes': list(pd.Series(labs_avg_cos).value_counts().sort_values(ascending=False).values),
    'silhouette': silhouette_score(emb_C_c, labs_avg_cos, metric='cosine'),
    'ari_cons_Cc': adjusted_rand_score(labs_avg_cos, cons_C_c),
    'ari_single_Cc': adjusted_rand_score(labs_avg_cos, df_runs['labels_C_c']),
    'noise_points': 0,
    'ari_cons_Cc_non_noise': adjusted_rand_score(labs_avg_cos, cons_C_c)
})

# c. GMM on 10 PCs
pca10 = PCA(n_components=10, random_state=42)
X_pca10 = pca10.fit_transform(emb_C_c)

bic_sph = {}
bic_diag = {}
labs_gmm_sph_7 = None
labs_gmm_diag_7 = None

for k in range(2, 13):
    gmm_s = GaussianMixture(n_components=k, covariance_type='spherical', n_init=20, random_state=42)
    gmm_s.fit(X_pca10)
    bic_sph[k] = gmm_s.bic(X_pca10)
    if k == 7: labs_gmm_sph_7 = gmm_s.predict(X_pca10)
    
    gmm_d = GaussianMixture(n_components=k, covariance_type='diag', n_init=20, random_state=42)
    gmm_d.fit(X_pca10)
    bic_diag[k] = gmm_d.bic(X_pca10)
    if k == 7: labs_gmm_diag_7 = gmm_d.predict(X_pca10)

best_k_sph = min(bic_sph, key=bic_sph.get)
best_k_diag = min(bic_diag, key=bic_diag.get)

print("=== GAUSSIAN MIXTURE MODEL BIC ACROSS k IN [2, 12] ===")
df_bic = pd.DataFrame({'k': list(range(2, 13)), 'BIC_spherical': [bic_sph[k] for k in range(2, 13)], 'BIC_diag': [bic_diag[k] for k in range(2, 13)]})
display(df_bic.round(2))
print(f"GMM Spherical: lowest BIC at k={best_k_sph} (BIC = {bic_sph[best_k_sph]:.2f})")
print(f"GMM Diag:      lowest BIC at k={best_k_diag} (BIC = {bic_diag[best_k_diag]:.2f})")

algo_results.append({
    'algorithm': 'GMM (Spherical, 10 PCs, k=7)',
    'k': 7,
    'cluster_sizes': list(pd.Series(labs_gmm_sph_7).value_counts().sort_values(ascending=False).values),
    'silhouette': silhouette_score(emb_C_c, labs_gmm_sph_7, metric='cosine'),
    'ari_cons_Cc': adjusted_rand_score(labs_gmm_sph_7, cons_C_c),
    'ari_single_Cc': adjusted_rand_score(labs_gmm_sph_7, df_runs['labels_C_c']),
    'noise_points': 0,
    'ari_cons_Cc_non_noise': adjusted_rand_score(labs_gmm_sph_7, cons_C_c)
})
algo_results.append({
    'algorithm': 'GMM (Diag, 10 PCs, k=7)',
    'k': 7,
    'cluster_sizes': list(pd.Series(labs_gmm_diag_7).value_counts().sort_values(ascending=False).values),
    'silhouette': silhouette_score(emb_C_c, labs_gmm_diag_7, metric='cosine'),
    'ari_cons_Cc': adjusted_rand_score(labs_gmm_diag_7, cons_C_c),
    'ari_single_Cc': adjusted_rand_score(labs_gmm_diag_7, df_runs['labels_C_c']),
    'noise_points': 0,
    'ari_cons_Cc_non_noise': adjusted_rand_score(labs_gmm_diag_7, cons_C_c)
})

# d. HDBSCAN min_cluster_size=5 and 8
for mcs in [5, 8]:
    hdb = HDBSCAN(min_cluster_size=mcs)
    labs_hdb = hdb.fit_predict(emb_C_c)
    n_clusters = len(set(labs_hdb) - {-1})
    n_noise = sum(labs_hdb == -1)
    
    non_noise_mask = (labs_hdb != -1)
    if n_clusters > 1 and sum(non_noise_mask) > n_clusters:
        sil_hdb = silhouette_score(emb_C_c[non_noise_mask], labs_hdb[non_noise_mask], metric='cosine')
    else:
        sil_hdb = np.nan
        
    ari_all_cons = adjusted_rand_score(labs_hdb, cons_C_c)
    ari_non_noise_cons = adjusted_rand_score(labs_hdb[non_noise_mask], cons_C_c[non_noise_mask]) if sum(non_noise_mask) > 0 else np.nan
    ari_all_single = adjusted_rand_score(labs_hdb, df_runs['labels_C_c'])
    ari_non_noise_single = adjusted_rand_score(labs_hdb[non_noise_mask], df_runs['labels_C_c'].values[non_noise_mask]) if sum(non_noise_mask) > 0 else np.nan
    
    algo_results.append({
        'algorithm': f'HDBSCAN (min_cluster_size={mcs})',
        'k': n_clusters,
        'cluster_sizes': list(pd.Series(labs_hdb[non_noise_mask]).value_counts().sort_values(ascending=False).values),
        'silhouette': sil_hdb,
        'ari_cons_Cc': ari_all_cons,
        'ari_single_Cc': ari_all_single,
        'noise_points': n_noise,
        'ari_cons_Cc_non_noise': ari_non_noise_cons
    })

df_algo_comparison = pd.DataFrame(algo_results)
print("\n=== ALGORITHM COMPARISON SUMMARY TABLE ===")
display(df_algo_comparison)


=== GAUSSIAN MIXTURE MODEL BIC ACROSS k IN [2, 12] ===


,k,BIC_spherical,BIC_diag
0,2,-393.99,-452.32
1,3,-459.03,-461.42
2,4,-483.44,-424.29
3,5,-501.31,-389.30
4,6,-515.03,-371.39
5,7,-524.57,-343.17
6,8,-534.54,-310.63
7,9,-532.60,-288.71
8,10,-529.97,-254.35
9,11,-520.44,-224.20


GMM Spherical: lowest BIC at k=8 (BIC = -534.54)
GMM Diag:      lowest BIC at k=3 (BIC = -461.42)

=== ALGORITHM COMPARISON SUMMARY TABLE ===


,algorithm,k,cluster_sizes,silhouette,ari_cons_Cc,ari_single_Cc,noise_points,ari_cons_Cc_non_noise
0,Agglomerative (Ward),7,"[36, 26, 26, 20, 17, 17, 16]",0.133363,0.542984,0.505871,0,0.542984
1,"Agglomerative (Average, Cosine)",7,"[41, 33, 24, 21, 20, 16, 3]",0.127614,0.580794,0.580340,0,0.580794
2,"GMM (Spherical, 10 PCs, k=7)",7,"[32, 31, 23, 21, 20, 18, 13]",0.150021,0.782355,0.755852,0,0.782355
3,"GMM (Diag, 10 PCs, k=7)",7,"[31, 28, 24, 24, 21, 19, 11]",0.150942,0.816539,0.790838,0,0.816539
4,HDBSCAN (min_cluster_size=5),2,"[110, 17]",0.099283,0.080041,0.075540,31,0.093216
5,HDBSCAN (min_cluster_size=8),3,"[61, 13, 10]",0.159371,0.118567,0.114676,74,0.265595


## 5. Alternative Embedding Baselines on Masked Text ($T_c$ and $M_c$)
Construct and evaluate two baseline embeddings on the masked text:
1. **$T_c$ (TF-IDF + SVD-50)**: Strip `[COUNTRY]`, compute sublinear TF-IDF, project to 50 dimensions via TruncatedSVD, L2-normalize, subtract column mean, and re-normalize.
2. **$M_c$ (all-MiniLM-L6-v2)**: Chunk masked text at $\le 200$ tokens, pool chunks via token-weighted mean, L2-normalize, subtract column mean, and re-normalize.

Compare $C_c$, $T_c$, and $M_c$ across:
- Cosine silhouette for $k \in [5, 9]$ ($n_{	ext{init}}=50, 	ext{random\_state}=42$).
- Seed stability at $k=7$ (30 seeds, $n_{	ext{init}}=10$: mean and min pairwise ARI).
- Consensus clustering ($200$ seeds) and $3 	imes 3$ consensus ARI agreement matrix.
- ARI against original clusters.

In [5]:
def mask_country_name(country, text):
    words = re.findall(r'[a-zA-Z]+', country)
    cap_words = [w for w in words if w and w[0].isupper() and len(w) >= 4]
    terms = set([country] + cap_words)
    sorted_terms = sorted(terms, key=len, reverse=True)
    pattern = r'\b(?:' + '|'.join(re.escape(t) for t in sorted_terms) + r')\b'
    masked_text, count = re.subn(pattern, '[COUNTRY]', text, flags=re.IGNORECASE)
    return masked_text, count

masked_texts = [mask_country_name(r['Country'], ' '.join(str(r['extracted_text']).split()))[0] for _, r in df_clusterable.iterrows()]

# 1. T_c: TF-IDF + SVD-50
untokenized_texts = [re.sub(r'\[COUNTRY\]', ' ', t) for t in masked_texts]
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.5, stop_words='english')
X_tfidf = tfidf.fit_transform(untokenized_texts)
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_tfidf)

norm1 = np.linalg.norm(X_svd, axis=1, keepdims=True)
norm1[norm1 == 0] = 1.0
X_svd_norm = X_svd / norm1
X_svd_c = X_svd_norm - X_svd_norm.mean(axis=0)
norm2 = np.linalg.norm(X_svd_c, axis=1, keepdims=True)
emb_T_c = X_svd_c / norm2
assert emb_T_c.shape == (158, 50)
print(f"Constructed emb_T_c with shape {emb_T_c.shape}")

# 2. M_c: all-MiniLM-L6-v2 (chunks <= 200 tokens)
model_mini = SentenceTransformer('all-MiniLM-L6-v2')
tok_mini = model_mini.tokenizer

def count_tokens_mini(text):
    return len(tok_mini.encode(str(text), add_special_tokens=False))

def chunk_mini(text, max_tokens=200):
    text = ' '.join(str(text).split())
    tot = count_tokens_mini(text)
    if tot <= max_tokens:
        return [text], [tot]
    raw_sents = [s.strip() for s in re.split(r'(?<=[.?!])\s+', text) if s.strip()]
    units = []
    for s in raw_sents:
        s_toks = tok_mini.encode(s, add_special_tokens=False)
        if len(s_toks) <= max_tokens:
            units.append(s)
        else:
            for j in range(0, len(s_toks), max_tokens):
                sub_ids = s_toks[j:j+max_tokens]
                sub_t = tok_mini.decode(sub_ids, skip_special_tokens=True).strip()
                if sub_t: units.append(sub_t)
    chunks, curr = [], []
    for u in units:
        cand = ' '.join(curr + [u]) if curr else u
        if count_tokens_mini(cand) <= max_tokens:
            curr.append(u)
        else:
            if curr: chunks.append(' '.join(curr))
            curr = [u]
    if curr: chunks.append(' '.join(curr))
    chunk_tokens = [count_tokens_mini(c) for c in chunks]
    return chunks, chunk_tokens

m_chunks, m_tokens = [], []
for mt in masked_texts:
    ch, tk = chunk_mini(mt)
    m_chunks.append(ch)
    m_tokens.append(tk)

all_m_chunks = []
m_counts = []
for c_list in m_chunks:
    all_m_chunks.extend(c_list)
    m_counts.append(len(c_list))

all_m_embs = model_mini.encode(all_m_chunks, batch_size=32, show_progress_bar=False, normalize_embeddings=False)
country_m_embs = []
idx = 0
for i, count in enumerate(m_counts):
    c_embs = all_m_embs[idx:idx+count]
    c_weights = np.array(m_tokens[i], dtype=np.float32)
    idx += count
    w_mean = np.average(c_embs, axis=0, weights=c_weights)
    norm = np.linalg.norm(w_mean)
    country_m_embs.append(w_mean / norm)

emb_M = np.array(country_m_embs, dtype=np.float32)
emb_M_c = center_and_normalize(emb_M)
assert emb_M_c.shape == (158, 384)
print(f"Constructed emb_M_c with shape {emb_M_c.shape}")

# Cosine silhouette k in 5..9
baseline_sils = {}
for name, emb in [('C_c', emb_C_c), ('T_c', emb_T_c), ('M_c', emb_M_c)]:
    baseline_sils[name] = {}
    for k in range(5, 10):
        km = KMeans(n_clusters=k, n_init=50, random_state=42)
        labs = km.fit_predict(emb)
        baseline_sils[name][k] = silhouette_score(emb, labs, metric='cosine')

print("\n=== BASELINE SILHOUETTE SCORES (COSINE, k=5..9) ===")
display(pd.DataFrame(baseline_sils).round(4))

# Seed stability at k=7
base_seed_stab = {}
for name, emb in [('C_c', emb_C_c), ('T_c', emb_T_c), ('M_c', emb_M_c)]:
    labs_seeds = [KMeans(n_clusters=7, n_init=10, random_state=s).fit_predict(emb) for s in range(30)]
    aris = [adjusted_rand_score(labs_seeds[i], labs_seeds[j]) for i, j in combinations(range(30), 2)]
    base_seed_stab[name] = {'mean_ari': np.mean(aris), 'min_ari': np.min(aris)}

print("\n=== BASELINE SEED STABILITY AT k=7 (30 SEEDS, n_init=10) ===")
display(pd.DataFrame(base_seed_stab).round(4))

# Consensus labels
cons_T_c, _, link_T, _ = get_consensus_labels(emb_T_c, 200)
cons_M_c, _, link_M, _ = get_consensus_labels(emb_M_c, 200)

base_cons_dict = {'C_c': cons_C_c, 'T_c': cons_T_c, 'M_c': cons_M_c}
ari_3x3_base = np.zeros((3, 3))
keys_3x3 = ['C_c', 'T_c', 'M_c']
for i, k1 in enumerate(keys_3x3):
    for j, k2 in enumerate(keys_3x3):
        ari_3x3_base[i, j] = adjusted_rand_score(base_cons_dict[k1], base_cons_dict[k2])

print("\n=== 3x3 ARI AGREEMENT MATRIX (CONSENSUS: C_c, T_c, M_c) ===")
display(pd.DataFrame(ari_3x3_base, index=keys_3x3, columns=keys_3x3).round(4))

print("\n=== ARI AGAINST ORIGINAL CLUSTERS ===")
for name in keys_3x3:
    ari_orig = adjusted_rand_score(base_cons_dict[name], df_order['orig_cluster'])
    print(f"  {name} consensus vs. orig_cluster ARI: {ari_orig:.4f}")


Token indices sequence length is longer than the specified maximum sequence length for this model (335 > 256). Running this sequence through the model will result in indexing errors


Constructed emb_T_c with shape (158, 50)


Constructed emb_M_c with shape (158, 384)



=== BASELINE SILHOUETTE SCORES (COSINE, k=5..9) ===


,C_c,T_c,M_c
5,0.1388,0.0994,0.1079
6,0.1515,0.1034,0.1199
7,0.1604,0.1035,0.1204
8,0.1535,0.1163,0.1189
9,0.1486,0.1035,0.1224



=== BASELINE SEED STABILITY AT k=7 (30 SEEDS, n_init=10) ===


,C_c,T_c,M_c
mean_ari,0.6866,0.3607,0.5489
min_ari,0.5095,0.2149,0.3425



=== 3x3 ARI AGREEMENT MATRIX (CONSENSUS: C_c, T_c, M_c) ===


,C_c,T_c,M_c
C_c,1.0000,0.3099,0.5552
T_c,0.3099,1.0000,0.2575
M_c,0.5552,0.2575,1.0000



=== ARI AGAINST ORIGINAL CLUSTERS ===
  C_c consensus vs. orig_cluster ARI: 0.4727
  T_c consensus vs. orig_cluster ARI: 0.2016
  M_c consensus vs. orig_cluster ARI: 0.4288


## 6. Chunk-Averaging Sensitivity Check ($D_c$)
Evaluate the impact of multi-chunk weighted averaging by constructing representation $D_c$: setting `model.max_seq_length = 300` on `all-mpnet-base-v2` and encoding the masked text directly without chunk averaging (truncating each text at 300 tokens). L2-normalize, column-center, and re-normalize.
Assess:
- Overall ARI between $D_c$ consensus and $C_c$ consensus.
- Restricted ARI on multi-chunk countries ($> 300$ tokens in Variant C) and on single-chunk countries ($\le 300$ tokens).
- Contingency crosstab of $C_c$ vs. $D_c$ for multi-chunk countries.
- Identification of the $C_c$ cluster with the highest median length and the proportion of its members retained together in $D_c$.

In [6]:
model_mpnet = SentenceTransformer('all-mpnet-base-v2')
model_mpnet.max_seq_length = 300

embs_d_raw = model_mpnet.encode(masked_texts, batch_size=32, show_progress_bar=False, normalize_embeddings=False)
norm_d = np.linalg.norm(embs_d_raw, axis=1, keepdims=True)
embs_d_norm = embs_d_raw / norm_d
embs_d_c = embs_d_norm - embs_d_norm.mean(axis=0)
emb_D_c = embs_d_c / np.linalg.norm(embs_d_c, axis=1, keepdims=True)
assert emb_D_c.shape == (158, 768)

cons_D_c, _, link_D, _ = get_consensus_labels(emb_D_c, 200)

tok_mpnet = model_mpnet.tokenizer
def count_tokens_mpnet(text):
    return len(tok_mpnet.encode(str(text), add_special_tokens=False))

c_tokens_count = [count_tokens_mpnet(t) for t in masked_texts]
is_multi_chunk = np.array([c > 300 for c in c_tokens_count])

print(f"Variant C Document Structure:")
print(f"  Multi-chunk countries (> 300 tokens):  {sum(is_multi_chunk)} / 158")
print(f"  Single-chunk countries (<= 300 tokens): {sum(~is_multi_chunk)} / 158")

ari_all_D = adjusted_rand_score(cons_C_c, cons_D_c)
ari_multi_D = adjusted_rand_score(cons_C_c[is_multi_chunk], cons_D_c[is_multi_chunk])
ari_single_D = adjusted_rand_score(cons_C_c[~is_multi_chunk], cons_D_c[~is_multi_chunk])

print(f"\nAgreement Between D_c Consensus (Truncated First Chunk) and C_c Consensus (Chunk-Averaged):")
print(f"  Overall ARI (all 158 countries):            {ari_all_D:.4f}")
print(f"  Restricted ARI on multi-chunk (n={sum(is_multi_chunk)}):     {ari_multi_D:.4f}")
print(f"  Restricted ARI on single-chunk (n={sum(~is_multi_chunk)}):    {ari_single_D:.4f}")

print("\n=== CROSSTAB: C_c CONSENSUS VS. D_c CONSENSUS (MULTI-CHUNK COUNTRIES ONLY) ===")
ct_multi = pd.crosstab(cons_C_c[is_multi_chunk], cons_D_c[is_multi_chunk], rownames=['C_c (Multi)'], colnames=['D_c (Multi)'])
display(ct_multi)

# C_c cluster with highest median extracted_words
highest_med_cluster = df_c_stats.loc[df_c_stats['median'].idxmax(), 'cluster']
highest_med_val = df_c_stats.loc[df_c_stats['median'].idxmax(), 'median']
print(f"\nC_c cluster with highest median extracted_words: Cluster {highest_med_cluster} (median = {highest_med_val} words)")

members_high = (cons_C_c == highest_med_cluster)
d_high_labs = cons_D_c[members_high]
d_high_counts = pd.Series(d_high_labs).value_counts()
print(f"Cluster {highest_med_cluster} members (n={sum(members_high)}) distribution in D_c:")
print(d_high_counts)
most_common_D = d_high_counts.iloc[0]
print(f"Largest co-clustered subset in D_c: {most_common_D} / {sum(members_high)} ({most_common_D/sum(members_high)*100:.1f}%)")


Token indices sequence length is longer than the specified maximum sequence length for this model (1501 > 384). Running this sequence through the model will result in indexing errors


Variant C Document Structure:
  Multi-chunk countries (> 300 tokens):  65 / 158
  Single-chunk countries (<= 300 tokens): 93 / 158

Agreement Between D_c Consensus (Truncated First Chunk) and C_c Consensus (Chunk-Averaged):
  Overall ARI (all 158 countries):            0.7689
  Restricted ARI on multi-chunk (n=65):     0.6878
  Restricted ARI on single-chunk (n=93):    0.8637

=== CROSSTAB: C_c CONSENSUS VS. D_c CONSENSUS (MULTI-CHUNK COUNTRIES ONLY) ===


D_c (Multi),0,1,2,3,4,5,6
C_c (Multi),,,,,,,
0,1,0,1,0,0,0,0
1,0,0,0,0,1,0,18
2,0,0,0,5,0,0,0
3,0,0,0,0,0,4,3
4,4,0,2,0,0,0,2
5,0,0,1,1,12,0,0
6,0,10,0,0,0,0,0



C_c cluster with highest median extracted_words: Cluster 5 (median = 479.0 words)
Cluster 5 members (n=16) distribution in D_c:
4    14
2     1
3     1
Name: count, dtype: int64
Largest co-clustered subset in D_c: 14 / 16 (87.5%)


## 7. Consensus Dataset Assembly and Export
Assemble all derived consensus partitions, confidence scores, runner-up clusters, and baseline consensus assignments into `data/processed/cluster_consensus.csv`.

In [7]:
df_consensus = df_order[['Country', 'status', 'extracted_words', 'orig_cluster']].copy()
df_consensus['cons_A'] = cons_labels_dict['A']
df_consensus['cons_B'] = cons_labels_dict['B']
df_consensus['cons_C'] = cons_labels_dict['C']
df_consensus['cons_A_c'] = cons_labels_dict['A_c']
df_consensus['cons_B_c'] = cons_labels_dict['B_c']
df_consensus['cons_C_c'] = cons_C_c
df_consensus['conf_C_c'] = conf_Cc
df_consensus['runnerup_C_c'] = runnerup_Cc
df_consensus['runnerup_score_C_c'] = runnerup_score_Cc
df_consensus['cons_T_c'] = cons_T_c
df_consensus['cons_M_c'] = cons_M_c
df_consensus['cons_D_c'] = cons_D_c

consensus_path = os.path.join(data_dir, 'cluster_consensus.csv')
df_consensus.to_csv(consensus_path, index=False)
print(f"Successfully saved consensus dataset to: {consensus_path}")
print(f"Dimensions: {df_consensus.shape}")
print(f"\nFirst 5 rows:")
display(df_consensus.head(5))


Successfully saved consensus dataset to: ../data/processed/cluster_consensus.csv
Dimensions: (158, 16)

First 5 rows:


,Country,status,extracted_words,orig_cluster,cons_A,cons_B,cons_C,cons_A_c,cons_B_c,cons_C_c,conf_C_c,runnerup_C_c,runnerup_score_C_c,cons_T_c,cons_M_c,cons_D_c
0,Afghanistan,preamble_ok,255,2,4,5,4,4,2,5,0.717000,6,0.324667,4,3,2
1,Albania,preamble_ok,114,1,2,1,6,2,5,2,0.934583,5,0.017500,1,2,3
2,Algeria,preamble_ok,1307,2,3,5,1,4,2,1,0.317692,5,0.659687,4,3,4
3,Andorra,preamble_ok,255,5,2,1,6,2,5,2,0.718542,4,0.178864,0,0,3
4,Angola,preamble_ok,580,4,3,6,1,6,4,1,0.875192,4,0.081136,4,4,6
